<!--
Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# **API Mapping Table: `torch.cuda` → `torch.tpu`**

Once the backend is initialized via `torch.device("tpu")`, TorchTPU registers a dedicated Python module: **`torch.tpu`**. This module is designed as a drop-in replacement for the standard `torch.cuda` API.

Use this reference to quickly translate CUDA-based code to TorchTPU.

## **Core Device Queries**

| CUDA API | TPU API | Description |
| :--- | :--- | :--- |
| `torch.cuda.is_available()` | `torch.tpu.is_available()` | Checks if the TPU backend is ready. |
| `torch.cuda.device_count()` | `torch.tpu.device_count()` | Number of devices assigned to the process. |
| `torch.cuda.current_device()` | `torch.tpu.current_device()` | Returns the active core index. |
| `torch.cuda.get_device_properties()` | *(Implicit in `torch.device("tpu")`)* | Hardware metadata via PjRt logs. |
| `torch.cuda.set_device(rank)` | Handled automatically by `torch.device("tpu")` | Backend init is automatic. |

In [1]:
import torch

# Initialize hardware
device = torch.device("tpu")

print(f"torch.tpu.is_available():    {torch.tpu.is_available()}")
print(f"torch.accelerator.is_available(): {torch.accelerator.is_available()}")
print(f"torch.tpu.device_count():    {torch.tpu.device_count()}")
print(f"torch.tpu.current_device():  {torch.tpu.current_device()}")

I0000 00:00:1787340632.105601   76442 pjrt_api.cc:119] GetPjrtApi was found for TPU at /home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/libtpu/libtpu.so
I0000 00:00:1787340632.122300   76442 tpu_profiler_plugin.cc:791] PyTorch PrivateUse1ProfilerRegistry not found. Native TPU profiling will be disabled (this is expected on PyTorch < 2.12).
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
I0000 00:00:1787340632.143411   76442 device_rt.cc:130] PjRt runtime initialization deferred for tpu
I0000 00:00:1787340632.146194   76442 pjrt_api.cc:168] The PJRT plugin has PJRT API version 0.114. The framework PJRT API version is 0.114.


Device type: tpu, Device index: default
Initializing TPU distributed runtime
torch.tpu.is_available():    True
torch.accelerator.is_available(): True
torch.tpu.device_count():    1
torch.tpu.current_device():  0


I0000 00:00:1787340633.476251   76442 pjrt_c_api_client.cc:188] PjRtCApiClient created.


## **Random Number Generation**

| CUDA API | TPU API | Description |
| :--- | :--- | :--- |
| `torch.cuda.manual_seed(seed)` | `torch.tpu.manual_seed(seed)` | Seeds the current core's RNG. |
| `torch.cuda.manual_seed_all(seed)` | `torch.tpu.manual_seed_all(seed)` | Seeds all TPU cores (for distributed). |

In [2]:
# Set the seed across all cores
torch.tpu.manual_seed_all(42)
print("Seeded all TPU cores with value 42.")

Seeded all TPU cores with value 42.


## **Automatic Mixed Precision (AMP)**

TorchTPU integrates with the standard `torch.amp.autocast` framework. The device string is `"tpu"`.

| CUDA API | TPU API | Description |
| :--- | :--- | :--- |
| `torch.amp.autocast("cuda", ...)` | `torch.amp.autocast("tpu", ...)` | Enables automatic mixed precision. |
| `torch.cuda.get_amp_supported_dtype()` | `torch.tpu.get_amp_supported_dtype()` | Lists hardware-accelerated dtypes. |

In [3]:
supported_dtypes = torch.tpu.get_amp_supported_dtype()
print(f"AMP supported dtypes: {supported_dtypes}")

AMP supported dtypes: [torch.bfloat16, torch.bool, torch.complex128, torch.complex64, torch.float16, torch.float32, torch.float64, torch.float8_e4m3fn, torch.float8_e4m3fnuz, torch.float8_e5m2, torch.float8_e5m2fnuz, torch.float8_e8m0fnu, torch.int16, torch.int32, torch.int64, torch.int8, torch.uint16, torch.uint32, torch.uint64, torch.uint8]


/tmp/ipykernel_76442/2213686486.py:1: UserWarning: get_amp_supported_dtype is experimental and subject to change: get_amp_supported_dtype() is experimental and may change or be removed without notice.
  supported_dtypes = torch.tpu.get_amp_supported_dtype()


## **Streams and Events**

The `torch.tpu` module provides `Stream` and `Event` objects to maintain API parity with CUDA.

| CUDA API | TPU API | Status |
| :--- | :--- | :--- |
| `torch.cuda.current_stream()` | `torch.tpu.current_stream()` | **Placeholder (dummy)** |
| `torch.cuda.default_stream()` | `torch.tpu.default_stream()` | **Placeholder (dummy)** |
| `torch.cuda.set_stream()` | `torch.tpu.set_stream()` | **Placeholder (dummy)** |

> [!NOTE]
> In TorchTPU's deferred execution model, the XLA compiler and PjRt runtime manage execution ordering and hardware concurrency automatically. Streams are not needed.

## **Cache and HBM Telemetry**

TorchTPU exposes specialized methods for auditing the Compilation Cache and hardware memory (HBM). These have **no CUDA equivalent**.

| TPU API | Description |
| :--- | :--- |
| `torch.tpu._hbm_usage_summary()` | HBM usage by compiled binaries. |
| `torch.tpu._get_cache_stats()` | Detailed cache hit/miss ratios and compilation latencies. |

In [4]:
# HBM usage summary (this is not a stable API, use with caution)
print(torch.tpu._hbm_usage_summary())

# Cache statistics (this is not a stable API, use with caution)
stats = torch.tpu._get_cache_stats()
print(f"\nCache Requests: {stats.num_cache_reqs}")
print(f"Cache Hits:     {stats.num_cache_hits}")

CompilationCache HBM usage: 0B for 0 executables.

Cache Requests: 0
Cache Hits:     0
